# Basic Logic Gates — From Boolean Functions to Waveforms

A logic gate maps one or more binary inputs to a single binary output through a fixed Boolean function. Every digital circuit, from an adder to a CPU, is a composition of the seven gates below plus the tri-state buffer.

$$Y = f(A, B), \qquad A, B, Y \in \{0, 1\}$$


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})


## The Seven Gates as Boolean Functions

Each gate is a pure function of its inputs. NAND and NOR are *universal*: any other gate can be built from copies of either one alone.

$$\text{NAND}(A,B)=\overline{A\cdot B}, \qquad \text{NOR}(A,B)=\overline{A+B}, \qquad \text{XOR}(A,B)=A\oplus B$$


In [2]:
def NOT(a):       return 1 - a
def AND(a, b):    return a & b
def OR(a, b):     return a | b
def NAND(a, b):   return 1 - (a & b)
def NOR(a, b):    return 1 - (a | b)
def XOR(a, b):    return a ^ b
def XNOR(a, b):   return 1 - (a ^ b)

GATES_2IN = {'AND': AND, 'OR': OR, 'NAND': NAND, 'NOR': NOR, 'XOR': XOR, 'XNOR': XNOR}

# sanity check: print the AND truth table
for a in (0, 1):
    for b in (0, 1):
        print(f'AND({a},{b}) = {AND(a,b)}')


AND(0,0) = 0
AND(0,1) = 0
AND(1,0) = 0
AND(1,1) = 1


## All Truth Tables Side by Side

Enumerating every input combination $(A,B)$ fully specifies a 2-input gate: there are $2^2 = 4$ input rows and $2^{2^2}=16$ possible 2-input functions, of which these six are the named ones.


In [3]:
inputs = [(0,0), (0,1), (1,0), (1,1)]
header = f"{'A':>2}{'B':>3} | " + ' '.join(f'{g:>5}' for g in GATES_2IN)
print(header)
print('-' * len(header))
for a, b in inputs:
    row = f'{a:>2}{b:>3} | ' + ' '.join(f'{GATES_2IN[g](a,b):>5}' for g in GATES_2IN)
    print(row)
print(f"\nNOT: NOT(0)={NOT(0)}  NOT(1)={NOT(1)}")


 A  B |   AND    OR  NAND   NOR   XOR  XNOR
-------------------------------------------
 0  0 |     0     0     1     1     0     1
 0  1 |     0     1     1     0     1     0
 1  0 |     0     1     1     0     1     0
 1  1 |     1     1     0     0     0     1

NOT: NOT(0)=1  NOT(1)=0


## Interactive Truth Table — Pick a Gate, Pick the Inputs

Select a gate and toggle $A,B$. The matching row of the truth table is highlighted and the output bit is shown large. This is the most direct way to internalise what each gate *decides*.


In [4]:
def show_truth(gate_name, A, B):
    fn = GATES_2IN[gate_name]
    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.axis('off')
    rows = [(0,0),(0,1),(1,0),(1,1)]
    ax.text(0.15, 1.02, 'A', weight='bold'); ax.text(0.4, 1.02, 'B', weight='bold')
    ax.text(0.7, 1.02, gate_name, weight='bold')
    for i, (a, b) in enumerate(rows):
        y = 0.85 - i * 0.22
        active = (a == A and b == B)
        if active:
            ax.axhspan(y - 0.06, y + 0.10, xmin=0.05, xmax=0.95,
                       color='#ffd24d', alpha=0.6, zorder=0)
        ax.text(0.15, y, str(a)); ax.text(0.4, y, str(b))
        ax.text(0.7, y, str(fn(a, b)),
                weight='bold' if active else 'normal')
    ax.text(0.5, -0.12, f'{gate_name}({A},{B}) = {fn(A,B)}',
            ha='center', fontsize=15, weight='bold', color='#c0392b')
    ax.set_xlim(0, 1); ax.set_ylim(-0.2, 1.1)
    plt.show()

w_gate = widgets.Dropdown(options=list(GATES_2IN), value='XOR', description='Gate:')
w_A = widgets.ToggleButtons(options=[0, 1], value=1, description='A:')
w_B = widgets.ToggleButtons(options=[0, 1], value=0, description='B:')
ui = widgets.VBox([w_gate, w_A, w_B])
out = widgets.interactive_output(show_truth, {'gate_name': w_gate, 'A': w_A, 'B': w_B})
display(ui, out)


Output()

## Gates Act on Waveforms Over Time

In a real circuit the inputs are not static: they are digital waveforms. A gate applies its Boolean function instant by instant. Below, two square-wave inputs of different periods are combined and the output waveform is drawn beneath them.


In [ ]:
def timing(gate_name, period_A, period_B):
    t = np.arange(0, 16)
    A = ((t // period_A) % 2).astype(int)
    B = ((t // period_B) % 2).astype(int)
    Y = np.array([GATES_2IN[gate_name](a, b) for a, b in zip(A, B)])

    fig, axes = plt.subplots(3, 1, figsize=(8, 4), sharex=True)
    for ax, sig, lab, c in zip(axes, [A, B, Y], ['A', 'B', f'Y = {gate_name}'],
                                ['#1f77b4', '#1f77b4', '#c0392b']):
        ax.step(t, sig, where='post', color=c, lw=2)
        ax.set_ylim(-0.3, 1.3); ax.set_yticks([0, 1])
        ax.set_ylabel(lab, rotation=0, ha='right', va='center')
    axes[-1].set_xlabel('clock tick')
    plt.tight_layout()
    plt.show()

w_g2 = widgets.Dropdown(options=list(GATES_2IN), value='AND', description='Gate:')
w_pA = widgets.IntSlider(value=2, min=1, max=6, description='period A:')
w_pB = widgets.IntSlider(value=3, min=1, max=6, description='period B:')
ui2 = widgets.VBox([w_g2, w_pA, w_pB])
out2 = widgets.interactive_output(timing, {'gate_name': w_g2, 'period_A': w_pA, 'period_B': w_pB})
display(ui2, out2)


Output()

## NAND Is Universal

Any Boolean function can be built from NAND gates alone. The three constructions below reproduce NOT, AND and OR; running them against the primitives confirms equality for all inputs.

$$\overline{A}=\text{NAND}(A,A), \quad A\cdot B=\text{NAND}\big(\text{NAND}(A,B),\,\text{NAND}(A,B)\big), \quad A+B=\text{NAND}(\overline{A},\overline{B})$$


In [6]:
def not_from_nand(a):    return NAND(a, a)
def and_from_nand(a, b): n = NAND(a, b); return NAND(n, n)
def or_from_nand(a, b):  return NAND(NAND(a, a), NAND(b, b))

checks = {'NOT': (not_from_nand, NOT, 1), 'AND': (and_from_nand, AND, 2), 'OR': (or_from_nand, OR, 2)}
for name, (built, ref, arity) in checks.items():
    if arity == 1:
        ok = all(built(a) == ref(a) for a in (0, 1))
    else:
        ok = all(built(a, b) == ref(a, b) for a in (0, 1) for b in (0, 1))
    print(f'{name} from NAND matches primitive: {ok}')


NOT from NAND matches primitive: True
AND from NAND matches primitive: True
OR from NAND matches primitive: True


## The Tri-State Buffer — A Third State

Unlike the logic gates above, a tri-state buffer has an *enable* line. When enabled it passes its input; when disabled it presents high impedance $Z$ — electrically disconnected, allowing many drivers to share one bus. We mark $Z$ at the midline.


In [7]:
def tri_state(enable_pattern):
    t = np.arange(0, 12)
    A = ((t // 2) % 2).astype(int)
    EN = np.array([int(c) for c in enable_pattern.ljust(12, '0')[:12]])
    fig, axes = plt.subplots(3, 1, figsize=(8, 4), sharex=True)
    axes[0].step(t, A, where='post', color='#1f77b4', lw=2)
    axes[0].set_ylabel('A', rotation=0, ha='right', va='center')
    axes[1].step(t, EN, where='post', color='#2ca02c', lw=2)
    axes[1].set_ylabel('EN', rotation=0, ha='right', va='center')
    for ax in axes[:2]:
        ax.set_ylim(-0.3, 1.3); ax.set_yticks([0, 1])
    axY = axes[2]
    for i in range(len(t)):
        if EN[i]:
            axY.plot([t[i], t[i]+1], [A[i], A[i]], color='#c0392b', lw=2.5)
        else:
            axY.plot([t[i], t[i]+1], [0.5, 0.5], color='gray', lw=1.5, ls=':')
            axY.text(t[i]+0.5, 0.62, 'Z', ha='center', color='gray', fontsize=8)
    axY.set_ylim(-0.3, 1.3); axY.set_yticks([0, 1])
    axY.set_ylabel('Y', rotation=0, ha='right', va='center')
    axY.set_xlabel('clock tick')
    plt.tight_layout()
    plt.show()

w_en = widgets.Text(value='110011001100', description='EN bits:',
                    layout=widgets.Layout(width='400px'))
out3 = widgets.interactive_output(tri_state, {'enable_pattern': w_en})
display(w_en, out3)


Text(value='110011001100', description='EN bits:', layout=Layout(width='400px'))

Output()

## Distinguishing OR, XOR and the Equality Gates

The most common confusion is OR vs XOR, and AND vs XNOR. The table maps input regimes to the gate that fires; the cell below verifies each claim programmatically.

| Inputs | OR | XOR | XNOR |
|--------|----|-----|------|
| both 0 | 0  | 0   | 1    |
| differ | 1  | 1   | 0    |
| both 1 | 1  | 0   | 1    |


In [8]:
claims = {
    'OR fires unless both 0':       all(OR(a,b)   == (1 if (a or b) else 0) for a in (0,1) for b in (0,1)),
    'XOR fires only when A != B':   all(XOR(a,b)  == (1 if a != b else 0)   for a in (0,1) for b in (0,1)),
    'XNOR fires only when A == B':  all(XNOR(a,b) == (1 if a == b else 0)   for a in (0,1) for b in (0,1)),
}
for desc, ok in claims.items():
    print(f'[{"PASS" if ok else "FAIL"}] {desc}')


[PASS] OR fires unless both 0
[PASS] XOR fires only when A != B
[PASS] XNOR fires only when A == B
